# Prevendo a Nota de Vinhos a partir de suas Características

### Projeto Final — Data Science

**Autores:** Bruno Vieira, Lucas Brun e Manuela Antunes

---

## 1. Introdução

A avaliação de vinhos por especialistas (sommeliers e críticos) é um processo
caro e subjetivo. A revista *Wine Enthusiast* atribui a cada vinho uma nota de
**80 a 100 pontos**. Será que conseguimos prever essa nota a partir de
características objetivas do vinho, como preço, país de origem, tipo de uva e a
própria descrição da resenha?

**Problema:** trata-se de um problema de regressão prever um valor numérico
contínuo (`points`) a partir de variáveis preditoras.

**Hipóteses iniciais que guiaram o trabalho:**

1. O **preço** é o preditor mais forte da nota vinhos mais caros tendem a
   receber notas mais altas.
2. O **país/região** de origem influencia a nota (terroir e tradição vitivinícola).
3. A relação preço–nota não é linear: aplicar o logaritmo do preço deve
   melhorar a correlação.

Ao longo do projeto testamos essas hipóteses com estatística e visualizações, e
comparamos três modelos de regressão para escolher o de melhor desempenho.

## 2. URL do Repositório no GitHub

🔗 **https://github.com/lucasbrun196/data_science_project**

## 3. Configuração do ambiente

> Envie o arquivo `winemag-data-130k-v2.csv` pelo painel de arquivos.

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42

DATA_PATH = "winemag-data-130k-v2.csv"

## 4. Dataset

**Origem:** *Wine Reviews* (Wine Enthusiast), disponível no Kaggle. São cerca de 130 mil resenhas de vinhos coletadas em 2017.

**Dicionário de variáveis:**

| Coluna | Descrição |
|---|---|
| `country` | País de origem do vinho |
| `description` | Texto da resenha do degustador |
| `designation` | Vinhedo dentro da vinícola |
| `points` | **Nota de 1 a 100 (alvo)** só são publicados vinhos com 80+ |
| `price` | Preço da garrafa (US\$) |
| `province` | Província/estado de origem |
| `region_1`, `region_2` | Regiões vitivinícolas mais específicas |
| `taster_name` | Nome do degustador |
| `title` | Título da resenha (costuma conter a safra) |
| `variety` | Tipo de uva (ex: Pinot Noir) |
| `winery` | Vinícola produtora |

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head(3)
df = df.drop(columns=["Unnamed: 0"], errors="ignore")   # remove só se existir
print("Dimensoes:", df.shape)
df.head(3)

In [ ]:
# Tipos e valores ausentes
print("Valores ausentes (%):")
print((df.isnull().mean() * 100).round(1).sort_values(ascending=False))

### 4.1. Limpeza dos dados

Decisões tomadas e suas justificativas:

- **Duplicatas:** o dataset tem milhares de resenhas idênticas. Removidas para
  não enviesar o modelo nem inflar artificialmente as métricas.
- **Valores ausentes em `price` (~7%):** o preço é o preditor mais importante,
  então optamos por remover essas linhas em vez de imputar um valor que
  distorceria a variável central. Ainda restam mais de 110 mil registros.
- **`country`, `province`, `variety` ausentes:** pouquíssimos casos, removidos.
- **`taster_name` ausente (~20%):** preenchido com a categoria `"Desconhecido"`.
- **Colunas descartadas** (`region_2`, `designation`, `taster_twitter_handle`):
  alto percentual de ausência e baixo valor preditivo adicional.

In [ ]:
df = df.drop_duplicates()
print("Apos remover duplicatas:", df.shape)

df = df.dropna(subset=["price", "country", "province", "variety"])
df["taster_name"] = df["taster_name"].fillna("Desconhecido")
print("Apos tratar ausentes essenciais:", df.shape)

### 4.2. Engenharia de features

- **`log_price`** = log(1 + preço). Comprime a forte assimetria do preço (poucos
  vinhos caríssimos) e, como veremos, melhora bastante a correlação com a nota.
- **`desc_length`** = número de palavras da resenha. Hipótese: resenhas mais
  longas/elaboradas podem estar associadas a vinhos mais notáveis.

In [ ]:
df["log_price"] = np.log1p(df["price"])
df["desc_length"] = df["description"].str.split().str.len()

FEATURES_NUM = ["log_price", "desc_length"]
FEATURES_CAT = ["country", "province", "variety", "winery", "taster_name"]
TARGET = "points"

df[FEATURES_NUM + [TARGET]].describe().round(2)

## 5. Análise Exploratória (EDA)

Investigamos os dados para testar as hipóteses da introdução.

In [ ]:
# Distribuicao do alvo
plt.figure(figsize=(7,4))
sns.histplot(df["points"], bins=21, kde=True, color="#7b2d43")
plt.title("Distribuicao das notas (points)"); plt.xlabel("points")
plt.show()

**Observação:** as notas se concentram entre **84 e 92**, com média de 88. A faixa
é estreita (80–100), o que limita o quanto de variância um modelo pode explicar,
ponto importante na hora de interpretar o R².

In [ ]:
# Hipotese 1: preco bruto vs log(preco)
fig, ax = plt.subplots(1, 2, figsize=(11,4))
sns.histplot(df["price"], bins=60, ax=ax[0], color="#7b2d43"); ax[0].set_xlim(0,300)
ax[0].set_title("price (bruto) - muito assimetrico")
sns.histplot(df["log_price"], bins=60, ax=ax[1], color="#2d6a7b")
ax[1].set_title("log(1+price) - quase simetrico")
plt.tight_layout(); plt.show()

print("Correlacao points x price      :", round(df["points"].corr(df["price"]), 3))
print("Correlacao points x log_price  :", round(df["points"].corr(df["log_price"]), 3))

**Confirma a Hipótese 1:** a correlação salta de **~0,42** (preço bruto) para
**~0,62** (log do preço). Isso justifica usar `log_price` na modelagem.

In [ ]:
# Hipotese 2: relacao preco-nota
amostra = df.sample(8000, random_state=RANDOM_STATE)
plt.figure(figsize=(7,4))
sns.regplot(data=amostra, x="log_price", y="points",
            scatter_kws={"alpha":0.15, "s":10, "color":"#7b2d43"},
            line_kws={"color":"black"})
plt.title("log(preco) vs nota"); plt.show()

In [ ]:
# Hipotese 3: nota media por pais (top 12 em volume)
top_c = df["country"].value_counts().head(12).index
g = (df[df["country"].isin(top_c)]
     .groupby("country")["points"].mean().sort_values(ascending=False))
plt.figure(figsize=(8,4))
sns.barplot(x=g.values, y=g.index, color="#7b2d43")
plt.xlim(85, 91); plt.xlabel("nota media")
plt.title("Nota media por pais (top 12 em volume)")
plt.show()
g.round(2)

In [ ]:
# Variedades de uva mais comuns
df["variety"].value_counts().head(10)

In [ ]:
# Mapa de correlacao das variaveis numericas
plt.figure(figsize=(4.5, 3.8))
sns.heatmap(df[["points","log_price","desc_length"]].corr(),
            annot=True, cmap="RdBu_r", vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlacao"); plt.tight_layout(); plt.show()

**Principais descobertas da EDA:**

1. **`log_price` é o preditor mais forte** (correlação ~0,62), confirmando as
   Hipóteses 1 e 2.
2. **`desc_length` também tem boa correlação** (~0,54): resenhas mais longas
   associam-se a notas maiores, uma feature valiosa.
3. **Há diferença de nota entre países** (Áustria e Alemanha no topo; Chile e
   Argentina abaixo), confirmando a Hipótese 3, justificando incluir variáveis
   geográficas no modelo.

## 6. Modelagem

**Separação treino/teste:** 80% / 20%.

**Tratamento das categóricas:** `country`, `province`, `variety`, `winery` e
`taster_name` têm cardinalidade muito alta (`winery` sozinha tem ~16 mil valores).
One-hot encoding criaria milhares de colunas. Usamos **Target Encoding**, que
substitui cada categoria pela média da nota associada a ela. O `TargetEncoder` do
scikit-learn faz isso com **validação cruzada interna**, evitando vazamento de dados.

**Modelos comparados:**
- **Regressão Linear**: baseline simples e interpretável.
- **Random Forest**: floresta de árvores, captura relações não-lineares.
- **Gradient Boosting** (`HistGradientBoostingRegressor`): boosting, costuma ser
  o mais preciso e é rápido.

In [ ]:
X = df[FEATURES_NUM + FEATURES_CAT]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Treino:", X_train.shape, "| Teste:", X_test.shape)

In [ ]:
def construir_preprocessador():
    return ColumnTransformer([
        ("num", StandardScaler(), FEATURES_NUM),
        ("cat", TargetEncoder(random_state=RANDOM_STATE), FEATURES_CAT),
    ])

modelos = {
    "Regressao Linear": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, max_depth=20, min_samples_leaf=4,
        max_samples=0.7, n_jobs=-1, random_state=RANDOM_STATE),
    "Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=400, learning_rate=0.08, random_state=RANDOM_STATE),
}

In [ ]:
resultados = []
modelos_treinados = {}

for nome, estimador in modelos.items():
    pipe = Pipeline([
        ("pre", construir_preprocessador()),
        ("model", estimador),
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    resultados.append({
        "Modelo": nome,
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "MAE":  mean_absolute_error(y_test, pred),
        "R2":   r2_score(y_test, pred),
    })
    modelos_treinados[nome] = pipe
    print(f"{nome:18s} treinado.")

tabela = (pd.DataFrame(resultados)
          .sort_values("R2", ascending=False)
          .reset_index(drop=True))
tabela.round(3)

## 7. Resultados e Discussão

In [ ]:
# Comparacao visual dos modelos
plt.figure(figsize=(7,4))
sns.barplot(data=tabela, x="R2", y="Modelo", color="#7b2d43")
plt.title("Comparacao de modelos (R2 - maior e melhor)")
plt.tight_layout(); plt.show()

melhor_nome = tabela.iloc[0]["Modelo"]
melhor = modelos_treinados[melhor_nome]
print("Melhor modelo:", melhor_nome)

In [ ]:
# Previsto vs Real (melhor modelo)
pred = melhor.predict(X_test)
plt.figure(figsize=(5,5))
plt.scatter(y_test, pred, alpha=0.1, s=8, color="#7b2d43")
plt.plot([80,100], [80,100], "k--")
plt.xlabel("nota real"); plt.ylabel("nota prevista")
plt.title(f"Previsto vs Real - {melhor_nome}")
plt.tight_layout(); plt.show()

In [ ]:
# Distribuicao dos residuos
residuos = y_test - pred
plt.figure(figsize=(7,4))
sns.histplot(residuos, bins=50, color="#2d6a7b")
plt.title("Distribuicao dos residuos (real - previsto)")
plt.xlabel("erro"); plt.show()
print("Erro medio:", round(residuos.mean(), 3),
      "| Desvio dos erros:", round(residuos.std(), 3))

In [ ]:
# Importancia das features (permutation importance no melhor modelo)
amostra_imp = X_test.sample(4000, random_state=RANDOM_STATE)
y_imp = y_test.loc[amostra_imp.index]
imp = permutation_importance(melhor, amostra_imp, y_imp,
                             n_repeats=5, random_state=RANDOM_STATE)
imp_df = (pd.DataFrame({"feature": X.columns, "importancia": imp.importances_mean})
          .sort_values("importancia", ascending=False))
plt.figure(figsize=(7,4))
sns.barplot(data=imp_df, x="importancia", y="feature", color="#7b2d43")
plt.title("Importancia das variaveis (permutation importance)")
plt.tight_layout(); plt.show()
imp_df.round(4)

**Discussão dos resultados:**

- O **Gradient Boosting** obteve o melhor desempenho: **RMSE ≈ 1,85**,
  **MAE ≈ 1,45** e **R² ≈ 0,65**.
- Na prática, o **MAE de ~1,45** significa que, em média, a nota prevista erra a
  real por menos de **1,5 ponto** numa escala de 100, um resultado bem preciso.
- O **R² de ~0,65** indica que o modelo explica cerca de 65% da variação das
  notas. Pode parecer modesto, mas é coerente: as notas variam pouco (80–100) e a
  qualidade percebida tem um componente subjetivo que nenhuma variável captura.
- A **importância das variáveis** confirma a EDA: `log_price` e `desc_length`
  dominam, seguidos pelas variáveis geográficas e de uva.
- Os **resíduos** são aproximadamente simétricos e centrados em zero, indicando que
  o modelo não tem viés sistemático grave.

In [ ]:
# Salvar o melhor modelo e as categorias para o dashboard
import os
os.makedirs("models", exist_ok=True)
joblib.dump(melhor, "models/best_model.joblib")
joblib.dump({c: sorted(df[c].dropna().unique().tolist()) for c in FEATURES_CAT},
            "models/categorias.joblib")
df[FEATURES_NUM + FEATURES_CAT + ["points","price"]].to_csv(
    "winemag-data-130k-v2.csv", index=False)
print("Modelo, categorias e dados limpos salvos.")

## 8. Dashboard Interativo

Dashboard construído com **Dash** e **Plotly Express**, rodando inline no Colab.
Utiliza o `df` já limpo e com as features criadas nas seções anteriores, sem reprocessamento.

**Filtros disponíveis:** variedade de uva, país, safra, faixa de nota e preço máximo.
Os gráficos atualizam simultaneamente a cada interação.

In [ ]:
# Instalar dependências do dashboard (caso ainda não instaladas)
!pip install dash

In [ ]:
df['vintage'] = (
    df['title']
    .str.extract(r'((?:19|20)\d{2})', expand=False)
    .astype('Int64')
)

In [ ]:
from dash import Dash, html, dcc, Output, Input
import plotly.express as px

# Extrai safra do título (caso ainda não tenha sido feito)
if 'vintage' not in df.columns:
    df['vintage'] = (
        df['title']
        .str.extract(r'((?:19|20)\d{2})', expand=False)
        .astype('Int64')
    )

# Opções dos filtros
top_varieties = ['Todas'] + df['variety'].value_counts().head(30).index.tolist()
countries     = ['Todos'] + sorted(df['country'].dropna().unique().tolist())
vintages      = ['Todas'] + [str(v) for v in sorted(
    df['vintage'].dropna().unique().astype(int).tolist(), reverse=True
)]

# Layout
app = Dash(__name__)

app.layout = html.Div([

    html.H2('🍷 Análise de Vinhos — Wine Enthusiast',
            style={'fontFamily': 'sans-serif', 'marginBottom': '4px'}),
    html.P('Dataset: Wine Reviews (Kaggle) · ~130 mil resenhas · Wine Enthusiast, 2017',
           style={'fontFamily': 'sans-serif', 'color': '#666',
                  'marginBottom': '20px', 'fontSize': '13px'}),

    # Filtros
    html.Div([

        html.Div([
            html.Label('Variedade (top 30)',
                       style={'fontFamily': 'sans-serif', 'fontSize': '13px'}),
            dcc.Dropdown(
                id='filtro-variety',
                options=[{'label': v, 'value': v} for v in top_varieties],
                value='Todas', clearable=False, style={'width': '200px'},
            ),
        ], style={'marginRight': '16px'}),

        html.Div([
            html.Label('País',
                       style={'fontFamily': 'sans-serif', 'fontSize': '13px'}),
            dcc.Dropdown(
                id='filtro-country',
                options=[{'label': c, 'value': c} for c in countries],
                value='Todos', clearable=False, style={'width': '160px'},
            ),
        ], style={'marginRight': '16px'}),

        html.Div([
            html.Label('Safra',
                       style={'fontFamily': 'sans-serif', 'fontSize': '13px'}),
            dcc.Dropdown(
                id='filtro-vintage',
                options=[{'label': v, 'value': v} for v in vintages],
                value='Todas', clearable=False, style={'width': '120px'},
            ),
        ], style={'marginRight': '32px'}),

        html.Div([
            html.Label('Faixa de nota',
                       style={'fontFamily': 'sans-serif', 'fontSize': '13px'}),
            dcc.RangeSlider(
                id='filtro-pts',
                min=int(df['points'].min()), max=int(df['points'].max()), step=1,
                value=[int(df['points'].min()), int(df['points'].max())],
                marks={80: '80', 85: '85', 90: '90', 95: '95', 100: '100'},
                tooltip={'placement': 'bottom', 'always_visible': False},
            ),
        ], style={'width': '260px', 'marginRight': '32px'}),

        html.Div([
            html.Label('Preço máx. (US$)',
                       style={'fontFamily': 'sans-serif', 'fontSize': '13px'}),
            dcc.Slider(
                id='filtro-price',
                min=5, max=int(df['price'].quantile(0.99)), step=5,
                value=int(df['price'].quantile(0.99)),
                tooltip={'placement': 'bottom', 'always_visible': False},
            ),
        ], style={'width': '200px'}),

    ], style={'display': 'flex', 'alignItems': 'flex-end',
              'flexWrap': 'wrap', 'gap': '8px', 'marginBottom': '20px'}),

    # KPIs
    html.Div(id='kpis', style={'display': 'flex', 'gap': '16px', 'marginBottom': '24px'}),

    # Linha 1: histograma + nota por país
    html.Div([
        dcc.Graph(id='hist-notas',  style={'flex': '1'}),
        dcc.Graph(id='bar-country', style={'flex': '1'}),
    ], style={'display': 'flex', 'gap': '16px', 'marginBottom': '16px'}),

    # Linha 2: dispersão preço × nota
    dcc.Graph(id='scatter-preco', style={'marginBottom': '16px'}),

    # Linha 3: top variedades + tendência por safra
    html.Div([
        dcc.Graph(id='bar-variety',  style={'flex': '1'}),
        dcc.Graph(id='line-vintage', style={'flex': '1'}),
    ], style={'display': 'flex', 'gap': '16px'}),

], style={'padding': '24px', 'fontFamily': 'sans-serif'})


# Callback
@app.callback(
    Output('kpis',          'children'),
    Output('hist-notas',    'figure'),
    Output('bar-country',   'figure'),
    Output('scatter-preco', 'figure'),
    Output('bar-variety',   'figure'),
    Output('line-vintage',  'figure'),
    Input('filtro-variety', 'value'),
    Input('filtro-country', 'value'),
    Input('filtro-vintage', 'value'),
    Input('filtro-pts',     'value'),
    Input('filtro-price',   'value'),
)
def atualizar(variety, country, vintage, pts_range, price_max):

    # Filtrar
    dff = df[
        (df['points'] >= pts_range[0]) &
        (df['points'] <= pts_range[1]) &
        (df['price']  <= price_max)
    ]
    if variety != 'Todas': dff = dff[dff['variety'] == variety]
    if country != 'Todos': dff = dff[dff['country'] == country]
    if vintage != 'Todas': dff = dff[dff['vintage'] == int(vintage)]

    # KPIs
    def kpi_card(label, value):
        return html.Div([
            html.P(label, style={'margin': '0 0 4px', 'fontSize': '12px', 'color': '#888'}),
            html.P(value, style={'margin': '0', 'fontSize': '22px', 'fontWeight': '500'}),
        ], style={
            'background': '#f5f5f5', 'borderRadius': '8px',
            'padding': '12px 20px', 'minWidth': '140px',
        })

    if len(dff) == 0:
        empty = px.scatter(title='Nenhum resultado para os filtros selecionados.')
        kpis  = [kpi_card(l, '—') for l in ['Vinhos','Nota média','Preço médio','Países','Variedades']]
        return kpis, empty, empty, empty, empty, empty

    kpis = [
        kpi_card('Vinhos',      f"{len(dff):,}".replace(',', '.')),
        kpi_card('Nota média',  f"{dff['points'].mean():.1f}"),
        kpi_card('Preço médio', f"US$ {dff['price'].mean():.0f}"),
        kpi_card('Países',      str(dff['country'].nunique())),
        kpi_card('Variedades',  str(dff['variety'].nunique())),
    ]

    # Histograma de notas
    fig_hist = px.histogram(
        dff, x='points', nbins=21,
        color_discrete_sequence=['#5DCAA5'],
        labels={'points': 'Nota', 'count': 'Vinhos'},
        title='Distribuição de notas', template='simple_white',
    )
    fig_hist.update_layout(showlegend=False, bargap=0.05)

    # Nota média por país
    top_c = dff['country'].value_counts().head(10).index
    avg_c = (
        dff[dff['country'].isin(top_c)]
        .groupby('country')['points'].mean()
        .sort_values(ascending=True).reset_index()
    )
    fig_country = px.bar(
        avg_c, x='points', y='country', orientation='h',
        color_discrete_sequence=['#7F77DD'],
        labels={'points': 'Nota média', 'country': ''},
        title='Nota média por país (top 10 em volume)',
        template='simple_white',
        range_x=[85, avg_c['points'].max() + 0.5],
    )

    # Dispersão preço × nota (amostra de até 5000)
    dff_s = dff.sample(min(5000, len(dff)), random_state=RANDOM_STATE).copy()
    dff_s['faixa'] = dff_s['points'].apply(
        lambda p: '90–100' if p >= 90 else ('85–89' if p >= 85 else '80–84')
    )
    fig_scatter = px.scatter(
        dff_s, x='log_price', y='points', color='faixa',
        color_discrete_map={'90–100': '#1D9E75', '85–89': '#EF9F27', '80–84': '#F0997B'},
        category_orders={'faixa': ['90–100', '85–89', '80–84']},
        hover_data={'title': True, 'variety': True, 'country': True,
                    'price': True, 'points': True, 'log_price': False, 'faixa': False},
        labels={'log_price': 'log(1 + preço)', 'points': 'Nota', 'faixa': 'Faixa'},
        title='Preço vs. Nota', opacity=0.5, template='simple_white',
    )
    fig_scatter.update_traces(marker=dict(size=5))

    # Top 10 variedades
    tv = (
        dff['variety'].value_counts().head(10)
        .reset_index().rename(columns={'variety': 'Variedade', 'count': 'Vinhos'})
    )
    fig_variety = px.bar(
        tv, x='Variedade', y='Vinhos', color='Variedade',
        color_discrete_sequence=[
            '#5DCAA5','#7F77DD','#EF9F27','#F0997B','#378ADD',
            '#D4537E','#639922','#D85A30','#85B7EB','#AFA9EC',
        ],
        title='Top 10 variedades por volume',
        template='simple_white', text='Vinhos',
    )
    fig_variety.update_traces(textposition='outside')
    fig_variety.update_layout(showlegend=False, xaxis_tickangle=30)

    # Tendência de nota por safra
    vt = (
        dff.dropna(subset=['vintage'])
        .groupby('vintage')['points']
        .agg(['mean', 'count']).reset_index()
        .rename(columns={'vintage': 'Safra', 'mean': 'Nota média', 'count': 'n'})
    )
    vt = vt[vt['n'] >= 10].sort_values('Safra')
    if len(vt) > 1:
        fig_vintage = px.line(
            vt, x='Safra', y='Nota média', markers=True,
            color_discrete_sequence=['#7F77DD'],
            hover_data={'n': True}, labels={'n': 'Vinhos'},
            title='Nota média por safra (safras com ≥10 vinhos)',
            template='simple_white',
        )
    else:
        fig_vintage = px.scatter(
            title='Dados insuficientes para exibir tendência por safra.',
            template='simple_white',
        )

    return kpis, fig_hist, fig_country, fig_scatter, fig_variety, fig_vintage


# Roda o servidor local — acesse http://localhost:8050
print('Dashboard disponível em: http://localhost:8050')
app.run(port=8050, debug=False, use_reloader=False)

## 9. Conclusão

Construímos um pipeline completo de Data Science para prever a nota de vinhos a
partir de características objetivas. As três hipóteses iniciais foram confirmadas:
o preço (em escala log) é o fator mais forte, a geografia influencia a nota, e a
relação preço–nota é não-linear. O **Gradient Boosting** previu as notas com erro
médio inferior a 1,5 ponto.

**Limitações:**
- As notas têm faixa estreita (80–100) e forte componente subjetivo, limitando o R².
- Não exploramos o **texto** das resenhas além do tamanho, técnicas de NLP
  (TF-IDF, embeddings) provavelmente elevariam o desempenho.
- Possível viés do degustador: cada crítico pode ter um "padrão" de notas.

**Trabalhos futuros:**
- Incorporar features de NLP a partir da `description`.
- Testar otimização de hiperparâmetros (GridSearch / Optuna).
- Extrair a **safra** do título e testá-la como preditor.